# Pulse 2026: Score Volatility vs Sample Size

This is the second notebook in the analysis sequence. It does not estimate the sample size needed
for a 95% confidence requirement; that answer is in `03-sample_size_reliability.ipynb`. It measures
the separate question of how much a country's reported IQB actually moves from month to month.

Two scatterplots. One point per country: how many measurements it collects in a typical month,
against how much its IQB score actually moved over the 24-month panel.

**Volatility is the median absolute deviation (MAD) of the country's observed monthly IQB series** —
half its months fall within that distance of its typical score. MAD rather than a standard
deviation because an IQB history is mostly a flat line with occasional jumps when a single
threshold test flips; an SD reports the jumps, MAD reports the typical state.

The two panels differ only in the percentile level the score is evaluated at: **p50** is the median
user in a country, **p90** a well-served one. Nothing else changes — the same measurements, the same
thresholds.

This notebook uses the reported percentiles directly. No pass-rate reconstruction is involved, so
nothing here depends on the modelling choices in `03-sample_size_reliability.ipynb`; see
`METHODOLOGY.md` for those.


## 1. Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pycountry
import pyarrow.parquet as pq

from iqb import IQBCalculator
from iqb.calculator.config import IQB_CONFIG

calculator = IQBCalculator()

DATA_DIR = Path("./data/v1")
OUT_DIR = Path("./analysis_outputs")
OUT_DIR.mkdir(exist_ok=True)

PERCENTILES = [1, 5, 10, 25, 50, 75, 90, 95, 99]
METRICS = ["download", "upload", "latency", "loss"]
LEVELS = [50, 90]          # the two reporting levels plotted
MIN_MONTHS = 12            # a MAD over fewer months than this is not a summary of anything

RAW_KEY = {"download": "download_throughput_mbps", "upload": "upload_throughput_mbps",
           "latency": "latency_ms", "loss": "packet_loss"}
HIGHER_BETTER = {"download": True, "upload": True, "latency": False, "loss": False}

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True,
                     "grid.alpha": 0.3, "grid.linestyle": "--"})
print(f"levels: {', '.join('p' + str(p) for p in LEVELS)}  |  "
      f"countries need >= {MIN_MONTHS} months to appear")

## 2. The Data

Same loader as notebook 03. Download and upload percentiles live in separate tables and are merged
on `country_code`. The x-axis is the **download-side** count: the `downloads_by_country` table
carries `download_*`, `latency_*` and `loss_*`, so seven of IQB's ten threshold tests rest on it.

In [ ]:
def discover_periods(cache_root: Path):
    out = []
    for period_dir in sorted(cache_root.iterdir()):
        if not period_dir.is_dir():
            continue
        for end_dir in sorted(period_dir.iterdir()):
            dl = end_dir / "downloads_by_country" / "data.parquet"
            ul = end_dir / "uploads_by_country" / "data.parquet"
            if dl.exists() and ul.exists():
                out.append((f"{period_dir.name[:4]}-{period_dir.name[4:6]}", dl, ul))
    return out


def load_all(cache_root: Path = DATA_DIR) -> pd.DataFrame:
    frames = []
    for month, dl_path, ul_path in discover_periods(cache_root):
        if dl_path.stat().st_size < 1000 or ul_path.stat().st_size < 1000:
            continue
        dl = pq.read_table(dl_path).to_pandas()
        ul = pq.read_table(ul_path).to_pandas()
        m = dl.merge(ul, on="country_code", suffixes=("_down", "_up"))
        m["n_down"] = m["sample_count_down"].astype(float)
        m["month"] = month
        frames.append(m)
    return pd.concat(frames, ignore_index=True)


def country_name(cc: str) -> str:
    c = pycountry.countries.get(alpha_2=cc)
    return c.name if c else cc


df = load_all()
print(f"{len(df):,} country-months  ({df.country_code.nunique()} countries x "
      f"{df.month.nunique()} months, {df.month.min()} .. {df.month.max()})")
print(f"download-side measurements per country-month: median {df.n_down.median():,.0f}  |  "
      f"P10 {df.n_down.quantile(.1):,.0f}  |  P90 {df.n_down.quantile(.9):,.0f}")

## 3. Scoring Every Country-Month at Both Levels

IQB flattens to ten weighted *(metric, threshold)* pass/fail tests — 24 use-case x requirement
combinations, but only 10 distinct tests, because use cases share thresholds. Scoring that way is
vectorised and exact; the cell below checks it against the production `IQBCalculator` before using
it 11,440 times.

In [ ]:
def derive_atoms(cfg=IQB_CONFIG):
    # Flatten the nested config into {metric: [(threshold, weight), ...]}.
    ucw_total = sum(uc["w"] for uc in cfg["use cases"].values())
    agg = {}
    for uc in cfg["use cases"].values():
        nrw_total = sum(nr["w"] for nr in uc["network requirements"].values())
        for nr_name, nr in uc["network requirements"].items():
            short = next(k for k, v in RAW_KEY.items() if v == nr_name)
            key = (short, nr["threshold min"])
            agg[key] = agg.get(key, 0.0) + (uc["w"] / ucw_total) * (nr["w"] / nrw_total)
    out = {m: [] for m in METRICS}
    for (m, thr), w in agg.items():
        out[m].append((thr, w))
    return out


ATOMS = derive_atoms()


def iqb_at_level(frame, level: int) -> np.ndarray:
    """IQB for every row, evaluated at the given percentile level."""
    score = np.zeros(len(frame))
    for m in METRICS:
        v = frame[f"{m}_p{level}"].to_numpy(float)
        for thr, w in ATOMS[m]:
            score += w * ((v > thr) if HIGHER_BETTER[m] else (v < thr))
    return score


# Verify the flattening against the production calculator before relying on it.
check = df.sample(300, random_state=0)
worst = max(
    abs(float(iqb_at_level(check.loc[[i]], p)[0])
        - calculator.calculate_iqb_score(
            data={"m-lab": {RAW_KEY[m]: float(r[f"{m}_p{p}"]) for m in METRICS}}))
    for i, r in check.iterrows() for p in LEVELS)
print(f"max |vectorised - IQBCalculator| over {len(check)} country-months x "
      f"{len(LEVELS)} levels: {worst:.2e}")

for p in LEVELS:
    df[f"iqb_p{p}"] = iqb_at_level(df, p)
print(f"\nmedian IQB across country-months:  " +
      "   ".join(f"p{p} {df[f'iqb_p{p}'].median():.3f}" for p in LEVELS))

## 4. Volatility Per Country

MAD of each country's monthly IQB series, in IQB units — *half of this country's months fell within
this distance of its typical score*. Notebook 03 §11 reports 1.4826x this figure, which rescales a
MAD to a standard-deviation equivalent for comparison against a predicted SD; here the raw value is
what is wanted.

In [ ]:
def mad(s: pd.Series) -> float:
    return float(np.median(np.abs(s - np.median(s))))


g = df.groupby("country_code")
vol = pd.DataFrame({
    "n_down_month_median": g.n_down.median(),
    "months": g.size(),
}).reset_index().rename(columns={"country_code": "cc"})
for p in LEVELS:
    vol[f"mad_iqb_p{p}"] = g[f"iqb_p{p}"].apply(mad).to_numpy()
    vol[f"median_iqb_p{p}"] = g[f"iqb_p{p}"].median().to_numpy()
vol["country"] = [country_name(c) for c in vol.cc]

dropped = int((vol.months < MIN_MONTHS).sum())
vol = vol[vol.months >= MIN_MONTHS].reset_index(drop=True)
print(f"{len(vol)} countries plotted ({dropped} dropped for fewer than {MIN_MONTHS} months)\n")

print(f"{'':>10s}{'median MAD':>13s}{'P90 MAD':>10s}{'MAD = 0':>12s}")
for p in LEVELS:
    c = vol[f"mad_iqb_p{p}"]
    print(f"{'p' + str(p):>10s}{c.median():>13.4f}{c.quantile(.9):>10.4f}"
          f"{(c == 0).mean():>11.0%}")

print(f"\n{'measurements / month':>22s}{'countries':>11s}" +
      "".join(f"{'median MAD p' + str(p):>19s}" for p in LEVELS))
bins = [0, 10, 100, 1_000, 10_000, 100_000, 1_000_000, 1e9]
labs = ["<10", "10-100", "100-1k", "1k-10k", "10k-100k", "100k-1M", ">1M"]
vol["bin"] = pd.cut(vol.n_down_month_median, bins, labels=labs)
for lab, sub in vol.groupby("bin", observed=True):
    print(f"{lab:>22s}{len(sub):>11d}" +
          "".join(f"{sub[f'mad_iqb_p{p}'].median():>19.4f}" for p in LEVELS))

## 5. The Two Scatterplots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.2), sharey=True)

for ax, p in zip(axes, LEVELS):
    y = vol[f"mad_iqb_p{p}"].to_numpy()
    x = vol.n_down_month_median.clip(lower=1).to_numpy()
    steady = y == 0

    ax.scatter(x[~steady], y[~steady], s=26, alpha=0.62, color="#2b6cb0",
               edgecolor="white", linewidth=0.4, label="moved between months", zorder=3)
    ax.scatter(x[steady], y[steady], s=26, alpha=0.55, color="#38a169", marker="D",
               edgecolor="white", linewidth=0.4,
               label=f"never moved (MAD = 0, n={steady.sum()})", zorder=3)

    # median MAD , to read the trend through the overplotting
    med = vol.groupby("bin", observed=True)[f"mad_iqb_p{p}"].median()
    ctr = vol.groupby("bin", observed=True).n_down_month_median.median()
    ax.plot(ctr, med, color="#c53030", lw=1.8, marker="o", ms=5, zorder=4,
            label="median")

    ax.set_xscale("log")
    ax.set_xlim(0.7, 1e8)
    ax.set_xlabel("download-side measurements per month (median)")
    ax.set_title(f"IQB evaluated at p{p}", fontsize=11)
    ax.legend(fontsize=8, loc="upper right", framealpha=0.92)

axes[0].set_ylabel("MAD of the country's monthly IQB score")
axes[0].set_ylim(-0.006, max(vol[[f"mad_iqb_p{p}" for p in LEVELS]].to_numpy().max() * 1.06, 0.05))
fig.suptitle("How much a country's published IQB score moves, against how much data it collects",
             fontsize=12)
fig.tight_layout()
plt.show()

## 6. What the Panels Show

In [ ]:
lo = vol[vol.n_down_month_median < 1_000]
hi = vol[vol.n_down_month_median >= 100_000]
print(f"{'':>26s}{'< 1k / month':>15s}{'>= 100k / month':>17s}")
print(f"{'countries':>26s}{len(lo):>15d}{len(hi):>17d}")
for p in LEVELS:
    print(f"{'median MAD at p' + str(p):>26s}"
          f"{lo[f'mad_iqb_p{p}'].median():>15.4f}{hi[f'mad_iqb_p{p}'].median():>17.4f}")
    print(f"{'share never moving':>26s}{(lo[f'mad_iqb_p{p}'] == 0).mean():>15.0%}"
          f"{(hi[f'mad_iqb_p{p}'] == 0).mean():>17.0%}")

out_path = OUT_DIR / "04_mad_vs_sample_size.csv"
cols = (["cc", "country", "n_down_month_median", "months"]
        + [f"{k}_iqb_p{p}" for p in LEVELS for k in ("median", "mad")])
vol.sort_values("n_down_month_median", ascending=False)[cols].to_csv(out_path, index=False)
print(f"\nwrote {out_path}")